In [16]:
import os
import sys
from pathlib import Path

Moving to project root

In [17]:
if Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.append(Path.cwd())

Finding the champion model and its versions

In [18]:
import mlflow
from mlflow import MlflowClient

In [19]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = MlflowClient()
MODEL_NAME = "turbofan-rul"

In [20]:
champ = client.get_model_version_by_alias(MODEL_NAME, "champion")
run = client.get_run(champ.run_id)

print(f"Champion model: {champ.version}")
print(f"Training window: {run.data.params["window"]}")

Champion model: 1
Training window: 10


Loading the model

In [21]:
model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{champ.version}")
print(type(model))

<class 'mlflow.pyfunc.PyFuncModel'>


Read the signature

In [22]:
from turbofan.features import FEATURE_SENSORS, get_feature_columns

schema = model.metadata.get_input_schema()
print(len(schema.input_names()), "names")
print("same columns and order as the package?", schema.input_names == get_feature_columns())


45 names
same columns and order as the package? False


In [25]:
for col in schema.inputs:
    if col.type.name != "double":
        print("not a float:", col.name, col.type.name)

print("FEATURE_SENSORS:", FEATURE_SENSORS)

not a float: sensor_17 long
FEATURE_SENSORS: ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


Real-time pred of one engine

In [32]:
from turbofan.data import RAW_DIR, load_cmapss, load_test_rul
from turbofan.features import add_rolling_features, last_cycle_per_engine

window = int(run.data.params["window"])

test_raw = load_cmapss(RAW_DIR / "test_FD001.txt")
engine = test_raw[test_raw["unit"] == 31]

feats = add_rolling_features(engine, window=window)
latest = last_cycle_per_engine(feats)

pred = model.predict(latest[get_feature_columns()])
truth = load_test_rul(RAW_DIR / "RUL_FD001.txt")

print("cycles of history:", len(engine))
print("Predicted RUL: ", round(float(pred[0]), 1))
print(truth[truth["unit"] == 31])


cycles of history: 196
Predicted RUL:  12.4
    rul  unit
30    8    31


Batch style, all engines

In [33]:
test_feats = last_cycle_per_engine(add_rolling_features(test_raw, window=window))
batch_preds = model.predict(test_feats[get_feature_columns()])

print(len(batch_preds), "predictions in one call")
print("engine 31 in the batch:", round(float(batch_preds[test_feats["unit"].values == 31][0]), 1))

100 predictions in one call
engine 31 in the batch: 12.4


Signature guarding the model

In [38]:
bad = latest[get_feature_columns()].drop(columns=["sensor_2"])

try:
    model.predict(bad)
except Exception as e:
    print(type(e).__name__, ":", str(e)[:300])

MlflowException : Failed to enforce schema of data '      sensor_3  sensor_4  sensor_6  ...  sensor_17_std  sensor_20_std  sensor_21_std
3869   1598.19   1420.66     21.61  ...       0.632456       0.108238       0.094017

[1 rows x 44 columns]' with schema '['sensor_2': double (required), 'sensor_3': double (require


Pydantic

What problem Pydantic solves

HTTP requests arrive as JSON text. Anything can be in it: a missing sensor, "hot" instead of a number, a negative cycle. Without validation, that garbage flows into pandas and fails somewhere deep with a confusing error, or worse, doesn’t fail and produces nonsense.

Pydantic lets you describe the expected shape as a Python class. It then checks incoming data against it, converts what can be safely converted, and lists everything wrong in one clear message.

In [40]:
from pydantic import BaseModel, Field, ValidationError

class Reading(BaseModel):
    cycle: int = Field(ge=1)
    sensor_2: float
    sensor_17: int

print(Reading(cycle=5, sensor_2=642.3, sensor_17=392))
print(Reading(cycle="5", sensor_2="642.3", sensor_17=392.0))

try:
    Reading(cycle=5, sensor_2="hot", sensor_17=392.8)
except ValidationError as e:
    print(e)

cycle=5 sensor_2=642.3 sensor_17=392
cycle=5 sensor_2=642.3 sensor_17=392
2 validation errors for Reading
sensor_2
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='hot', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/float_parsing
sensor_17
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=392.8, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/int_from_float


A request with history plus a custom rule

In [43]:
from pydantic import field_validator

class History(BaseModel):
    unit: int = Field(ge=1)
    cycles: list[Reading] = Field(min_length=1)

    @field_validator("cycles")
    @classmethod
    def cycles_are_unique(cls, cycles):
        numbers = [c.cycle for c in cycles]
        if len(numbers) != len(set(numbers)):
            raise ValueError("cycle numbers must be unique")
        return cycles

ok = History(unit=31, cycles=[
    {"cycle": 1, "sensor_2": 642.1, "sensor_17": 391},
    {"cycle": 2, "sensor_2": 642.4, "sensor_17": 392},
])
print(ok.cycles[1])

try:
    History(unit=31, cycles=[
        {"cycle": 1, "sensor_2": 642.1, "sensor_17": 391},
        {"cycle": 1, "sensor_2": 642.4, "sensor_17": 392},
    ])
except ValidationError as e:
    print(e)

cycle=2 sensor_2=642.4 sensor_17=392
1 validation error for History
cycles
  Value error, cycle numbers must be unique [type=value_error, input_value=[{'cycle': 1, 'sensor_2':...42.4, 'sensor_17': 392}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error


From Validated request to DF

In [44]:
import pandas as pd

history = pd.DataFrame([c.model_dump() for c in ok.cycles])
history["unit"] = ok.unit
print(history.dtypes)

cycle          int64
sensor_2     float64
sensor_17      int64
unit           int64
dtype: object


Calling the serve model from notebook

In [ ]:
import httpx

cols = ["cycle"] + FEATURE_SENSORS
payload = {"unit": 31, "cycles": engine[cols].to_dict(orient="records")} # to_dict converts df to dict

r = httpx.post("http://localhost:8000/predict", json=payload)
print(r.status_code)
print(r.json)

200
<bound method Response.json of <Response [200 OK]>>
